# Individual-business GAM-style and CatBoost comparison

## tl;dr

This experiment extends notebook [03](03_location_forecast_poc.ipynb) at the **individual business/calendar-day** grain. It compares the same two calendar baselines, identity-aware Poisson and HGB, plus an additive spline Poisson model and categorical-identity CatBoost. Scores are recalculated from the same source panels; no old ZIP scores or incompatible cached metrics are combined.

Start Jupyter with `uv run --frozen --env-file .env --group notebooks jupyter lab` after the imported database and notebook 03 are ready. All findings below come from this execution. This research experiment does not promote a model into the app.

### 1. Environment and required CatBoost dependency

In [ ]:
import importlib.metadata
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from sitesense.business_research import (
    ExperimentConfig,
    latest_run,
    load_models,
    load_run,
    recommend,
    run_experiment,
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 16)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})
display(
    pd.Series(
        {
            package: importlib.metadata.version(package)
            for package in ["sitesense", "numpy", "pandas", "scikit-learn"]
        },
        name="Installed version",
    )
)
print("CatBoost:", importlib.metadata.version("catboost"))

## Context & Methods

The target is **recorded check-ins for one business on one calendar day**, including zero-activity days within declared imported coverage. There is no ZIP aggregation, division by cohort size, multiplication by current listing count, or allocation of an area prediction to businesses.

| Stage | Fit interval | Assessment interval |
|---|---|---|
| Select fixed model specification | 2017–2019 | 2020 validation |
| Independent forward assessment | 2018–2020 | 2021 test |
| Refit saved research estimators | 2019–2021 | Future dates only |

Each stage rebuilds its own cohort using `first_activity <= fit_start` and at least **20 training check-ins**. No held-out activity, listing status, review count, rating, or future `last_checkin` determines eligibility. Model choice is frozen on 2020 mean daily Poisson deviance **before** assessing 2021. The test remains descriptive evidence, not another selection round.

### Key assumptions

- Scope is exact canonical city/state → parent category → optional exact source category. Source tags overlap; each listing contributes its **whole** recorded activity once within the selected source cohort. Snapshot tags and coordinates label an archived cohort, not historical point-in-time membership.
- To bound runtime, select at most `MAX_BUSINESSES` eligible IDs by deterministic hash of the seed and business ID, independently for each fit. The default is 64 and the cap is 128. Show full eligible and sample counts. Results describe that sample, not a citywide market.
- Calendar shrinkage priors use the complete supported parent/city pool through the production `business_training.fit` function **before** sampling the selected source cohort.
- Weather models all fit and score the same complete accepted mapped ACIS station/day reports: minimum and maximum temperature in °C, precipitation in mm. Trace rain is zero; rejected or absent observations remain missing. No scenario fills missing historical weather.
- Calendar baselines learn all covered training days; weather candidates learn only accepted-weather training rows. The same-held-out-row comparison therefore compares candidate systems and does **not** isolate weather uplift by itself.
- Calendar baselines also score **all covered business/day rows**, including days without weather. Compare weather candidates only on the shared `accepted_weather` population and inspect the missing-weather counts alongside it.
- Every learned model includes business identity: one-hot GLM effects, categorical HGB identity, or categorical CatBoost identity. Encoders, scaling and spline knots fit training rows only. No activity lags or snapshot profile values enter features.

Check-ins are an activity proxy. Observed target-day weather permits retrospective conditional prediction; an operational future prediction needs weather known at its origin. An explicit scenario is conditional evidence, not a verified weather forecast or a causal weather effect.

The 2020/2021 periods were previously examined in earlier project work; these forward results remain exploratory.

## Data

### 2. Load the most recent complete notebook 03 run

Set `POC_RUN` to a specific directory to reproduce an earlier compatible run; otherwise discover the latest **complete individual-business** run. Verification includes current source/catalog, protocol, source panels, code and lockfile plus all payload hashes. The configuration is inherited, not retyped. Any change requires rerunning notebook 03 first.

In [ ]:
POC_RUN = None  # Optional explicit Path to a complete business-level 03 run.
poc_directory = (
    Path(POC_RUN) if POC_RUN else latest_run(ROOT / "data/processed/business_forecast_poc")
)
poc = load_run(poc_directory)
CONFIG = ExperimentConfig(**poc.manifest["config"])
MODELS = ("overall_mean", "month_weekday_shrunk", "poisson", "hgb", "gam_poisson", "catboost")
display(pd.Series(CONFIG.__dict__, name="Inherited exact scope"))
print("Verified parent:", poc.directory.relative_to(ROOT))

### 3. Refit all candidates on compatible panels

`gam_poisson` is a **GAM-style additive Poisson GLM**, implemented with training-fitted cubic weather spline bases plus business/month/weekday effects. It is not an automatically tuned GAM package. The linear GLM and spline GLM both use fixed alpha 0.01. HGB and CatBoost use fixed specifications and no random held-out early-stopping split. CatBoost uses its native categorical business ID. Neither snapshot profile fields nor target-derived lags are inputs.

The saved parent manifest hash fixes lineage. The shared engine requires identical source/catalog/config/protocol and stage panel signatures, then reruns all six candidates and selects using validation only. Full-parent calendar priors remain outside the small source sample.

In [ ]:
run = run_experiment(
    CONFIG,
    MODELS,
    ROOT / "data/processed/business_model_comparison",
    parent_run=poc.directory,
)
run = load_run(run.directory)
print("Verified comparison:", run.directory.relative_to(ROOT))
display(pd.DataFrame.from_dict(run.manifest["stages"], orient="index").drop(columns="business_ids"))
assert run.manifest["panel_signatures"] == poc.manifest["panel_signatures"]
assert run.manifest["source_signature"] == poc.manifest["source_signature"]
assert run.manifest["catalog_signature"] == poc.manifest["catalog_signature"]

## Results

### 4. Compare validation selection and the forward exploratory assessment

The shared `accepted_weather` rows determine the comparison. Inspect the separate `all_covered` calendar scores and stage missing-weather counts so weather filtering cannot hide difficult covered days. An identical source cohort does not imply identical listing IDs across independently rebuilt fit windows.

In [ ]:
display(run.metrics.sort_values(["stage", "population", "poisson_deviance"]))
selected_model = run.manifest["selected_model"]
display(
    Markdown(
        f"**2020 validation selection:** `{selected_model}`. "
        "2021 is a forward assessment of the fixed candidates, not a tuning set."
    )
)

### 5. Plot same-row errors

In [ ]:
comparison = run.metrics.query("population == 'accepted_weather'").copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for stage, frame in comparison.groupby("stage", sort=False):
    axes[0].plot(frame.model, frame.poisson_deviance, marker="o", label=stage)
    axes[1].plot(frame.model, frame.mae, marker="o", label=stage)
axes[0].set(title="Same accepted-weather rows", ylabel="Mean daily Poisson deviance")
axes[1].set(title="Individual business/day counts", ylabel="MAE in recorded check-ins")
for ax in axes:
    ax.tick_params(axis="x", rotation=40)
    ax.legend()
plt.tight_layout()
plt.show()

### 6. Check that rerun baseline metrics reproduce notebook 03

This check uses the saved parent's full precision metrics within floating-point tolerance. It compares compatible newly calculated baseline results; it never imports parent scores into the new model contest.

In [ ]:
shared = list(poc.manifest["model_names"])
old = poc.metrics.set_index(["stage", "model", "population"]).sort_index()
new = (
    run.metrics.loc[run.metrics.model.isin(shared)]
    .set_index(["stage", "model", "population"])
    .sort_index()
)
pd.testing.assert_frame_equal(old, new, check_exact=False, rtol=1e-8, atol=1e-10)
print("All shared candidate metrics reproduced on the same business/day panels.")

### 7. Inspect individual competitors and error concentration

These are recorded listings in the selected niche, not ZIP centers. Accepted-weather rankings use each business's available assessment-day exposure; per-business observation counts make uneven station coverage visible. The cohort sample cannot support citywide top-business or opening-success claims.

In [ ]:
chosen = run.manifest["selected_model"]
ranking = run.ranking.query(
    "stage == 'test' and population == 'accepted_weather' and model == @chosen"
).sort_values("predicted_rank")
profiles = pd.read_json(run.directory / "catalog.json")[["business_id", "name"]]
display(ranking.merge(profiles, on="business_id", validate="one_to_one").head(10))
display(
    run.business_metrics.query(
        "stage == 'test' and population == 'accepted_weather' and model == @chosen"
    )
    .sort_values("mae", ascending=False)
    .head(10)
)

### 8. Inspect the actual persisted model specifications

In [ ]:
saved_models = load_models(run.directory)
display(
    pd.DataFrame(
        [
            {
                "model": name,
                "supported_businesses": len(model.business_ids),
                "weather_required": model.weather_features,
                "parameters": model.parameters,
            }
            for name, model in saved_models.items()
        ]
    )
)
assert set(saved_models) == set(MODELS)
assert all(len(model.business_ids) <= CONFIG.max_businesses for model in saved_models.values())

### 9. Compare conditional future rankings

This explicit common weather scenario is in °C/mm. Final models support only their 2019–2021 eligible businesses after that training cutoff. There is no prediction for an unseen business or allocation from an area score. Different weather conditions are conditional what-if inputs, not causal interventions.

In [ ]:
FUTURE_DAY = date(2022, 7, 1)
WEATHER_SCENARIO = {"tmin_c": 20.0, "tmax_c": 30.0, "precipitation_mm": 0.0}
# Calendar models do not require weather. Learned weather models require an explicit
# scenario here, or complete accepted observed reports for a retrospective day.
display(recommend(run.directory, FUTURE_DAY, model="month_weekday_shrunk", top_n=10))
display(recommend(run.directory, FUTURE_DAY, model=chosen, weather=WEATHER_SCENARIO, top_n=10))

## Takeaways

The dynamically generated summary reports the winner chosen from 2020 and its separate 2021 error. A more flexible candidate is useful only if its forward evidence supports it; no fixed winner is embedded in this notebook. Continue to [06](06_hgb_model_selection_report.ipynb) for a report using this exact parent/child lineage and [07](07_hgb_for_beginners.ipynb) for the individual-business HGB walkthrough.

In [ ]:
chosen_metrics = run.metrics.query("model == @selected_model and population == 'accepted_weather'")[
    ["stage", "businesses", "observations", "poisson_deviance", "mae"]
]
display(chosen_metrics)
display(
    Markdown(
        f"Validation chose **{selected_model}** from {len(MODELS)} fixed candidates. "
        "The displayed 2021 test is independent of model selection. "
        "Results describe a deterministic sample within one exact source category, "
        "not the full metro market, new-site demand, revenue, or causal weather effects. "
        "The production app continues to use its separately verified calendar bundle."
    )
)